# AUSA attorney tracker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/ausa-attorney-tracker/blob/main/ausa_attorney_tracker.ipynb)

Monthly career-attorney headcount for Assistant U.S. Attorneys (DOJ, Executive Office for U.S. Attorneys and the Offices of the U.S. Attorneys, occupational series 0905), queried **live** from the public OPM/EHRI mirror on HuggingFace ([`impactproject/opm-ehri-data`](https://huggingface.co/datasets/impactproject/opm-ehri-data)) with DuckDB over HTTPS. Nothing is downloaded to disk.

**Employment (headcount) only, not accessions/separations.** OPM's own guidance is that when the flow data (accessions/separations, keyed to each action's effective date) disagrees with the employment snapshot (keyed to snapshot date) — which happens, sometimes by a lot in a single month — the employment snapshot is authoritative. Confirmed directly: DC's accessions-minus-separations net for Jan 2025 was -30 (-5.6% of headcount), but DC's actual headcount barely moved that month (531→528); most of the drop landed in February's snapshot instead (528→492). This notebook only queries employment, per that guidance, rather than publish a flow-based number that wouldn't match the headcount someone could check directly.

**Scope: DC vs. rest-of-country, not state-by-state.** Every geographic field in this data (`duty_station_state_abbreviation`, `duty_station_city`, `core_based_statistical_area`) is privacy-redacted for ~91% of AUSA records — a small-occupational-subgroup suppression rule applied uniformly across the whole location hierarchy. DC is the one exception, since its ~500-attorney cell is large enough to clear the suppression threshold. So the only two honest "area" buckets available are **DC** and **rest-of-country (aggregate)** — this notebook does not fabricate state-level detail the data doesn't actually contain.

In [ ]:
# Setup — installs duckdb/pandas/great_tables if missing (all pip-installable on Colab)
for pkg, mod in [("duckdb", "duckdb"), ("pandas", "pandas"), ("great_tables", "great_tables")]:
    try:
        __import__(mod)
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import duckdb
import pandas as pd
from great_tables import GT
import urllib.request
import json
import re

In [ ]:
# Config
REPO = "impactproject/opm-ehri-data"
HF = f"https://huggingface.co/datasets/{REPO}/resolve/main/"
AGENCY_SUBELEMENT = "EXECUTIVE OFFICE FOR U.S. ATTORNEYS AND THE OFFICES OF THE U.S. ATTORNEYS"
SERIES_CODE = "0905"  # attorney

# Nov 2024 (pre-inauguration baseline) through present. Employment snapshots
# are the full federal workforce each month (26-75 MiB per file) filtered
# down to ~6,000 AUSA rows -- within this narrow ~20-month window, pulling
# every single one is still fast (~20s) and doesn't trip HuggingFace's rate
# limit (that only happened pulling the FULL 2015-present history, ~140
# employment files, in an earlier version of this notebook).
# EMPLOYMENT_MONTH_STRIDE is a fixed, deterministic interval (not a
# statistical sample) -- raise it above 1 if you widen START_YM/END_YM back
# toward full history and want to keep the query fast.
START_YM = "202411"
END_YM = "202612"
EMPLOYMENT_MONTH_STRIDE = 1  # 1 = every available month

# appointment_type values that mean "political appointee," excluded below so
# the table tracks the career AUSA workforce. Confirmed by enumerating every
# distinct appointment_type actually present for this population Nov 2024-
# present (2026-08-08) and checking each one, not guessed:
#   - SCHEDULE C: the standard political-appointee schedule (5 CFR 213.3301)
#   - NONCAREER (SENIOR EXECUTIVE SERVICE PERMANENT): noncareer SES = political
#   - EXECUTIVE (EXCEPTED SERVICE NONPERMANENT): always pay_plan_code='AD',
#     grade='40', supervisory_status='SUPERVISOR OR MANAGER' -- one specific,
#     consistent combination, consistent with the (Presidentially-appointed,
#     Senate-confirmed) U.S. Attorney / top leadership slot per district, not
#     an ordinary career attorney classification.
# NOTE: "OTHER (EXCEPTED SERVICE NONPERMANENT)" is NOT excluded even though
# "NONPERMANENT" sounds temporary -- checked back to 2015 and it's the
# dominant code for ordinary AUSA hires in every year, not a marker of
# temporary/surge staffing specific to this window.
POLITICAL_APPOINTMENT_TYPES = [
    "SCHEDULE C (EXCEPTED SERVICE NONPERMANENT)",
    "NONCAREER (SENIOR EXECUTIVE SERVICE PERMANENT)",
    "EXECUTIVE (EXCEPTED SERVICE NONPERMANENT)",
]

In [ ]:
def list_all_files(repo=REPO):
    """Every file in the HF tree, following pagination.

    The tree API caps a single response at 1000 entries (Link header,
    rel="next", cursor-based). This repo already has 1000+ files across
    accessions/employment/separations combined — a one-shot fetch silently
    truncates whichever directory sorts last alphabetically (separations/).
    """
    url = f"https://huggingface.co/api/datasets/{repo}/tree/main?recursive=true&limit=1000"
    out = []
    while url:
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req) as r:
            headers = dict(r.getheaders())
            out.extend(json.load(r))
        link = headers.get("Link")
        m = re.search(r'<([^>]+)>;\s*rel="next"', link) if link else None
        url = m.group(1) if m else None
    return out


def monthly_urls(files, dataset, start, end):
    """Latest version per month, from `start` through `end` (YYYYMM strings)."""
    best = {}
    for f in files:
        m = re.search(dataset + r"_(\d{6})_v(\d+)\.parquet", f["path"])
        if not m:
            continue
        month, ver = m.group(1), int(m.group(2))
        if start <= month <= end and (month not in best or ver > best[month][0]):
            best[month] = (ver, f["path"])
    return [HF + best[m][1] for m in sorted(best)]


files = list_all_files()
print(f"{len(files)} files total in the HF repo")

In [ ]:
con = duckdb.connect()
con.execute("SET enable_progress_bar=false;")
con.execute("INSTALL httpfs; LOAD httpfs;")
# Safety net, not the primary defense — the real fix against HuggingFace's
# rate limit is querying far fewer/smaller files in the first place (see
# START_YM/EMPLOYMENT_MONTH_STRIDE above). If this cell still errors with
# HTTP 429 after all retries, just re-run it — it's a temporary throttle.
con.execute("SET http_retries=6;")
con.execute("SET http_retry_wait_ms=1000;")
con.execute("SET http_retry_backoff=2;")
con.execute("SET threads=4;")

AREA_CASE = "CASE WHEN duty_station_state_abbreviation='DC' THEN 'DC' ELSE 'Rest of country' END"
POLITICAL_EXCLUSION_SQL = "(" + ",".join(f"'{t}'" for t in POLITICAL_APPOINTMENT_TYPES) + ")"


def query_employment(stride=EMPLOYMENT_MONTH_STRIDE):
    """Monthly (ym, area, headcount) totals for the career AUSA population.

    `stride` pulls every Nth available employment file instead of every one
    -- a fixed, deterministic interval, not a statistical sample. Filters
    on `snapshot_yyyymm BETWEEN START_YM AND END_YM` explicitly, not just
    on which files get fetched, and excludes POLITICAL_APPOINTMENT_TYPES
    (see config cell) so headcount figures reflect the career AUSA
    workforce, not political leadership turnover.
    """
    urls = monthly_urls(files, "employment", START_YM, END_YM)[::stride]
    lst = "[" + ",".join(f"'{u}'" for u in urls) + "]"
    return con.execute(f"""
        SELECT snapshot_yyyymm AS ym,
               {AREA_CASE} AS area,
               SUM(TRY_CAST(count AS BIGINT)) AS headcount
        FROM read_parquet({lst}, union_by_name=true)
        WHERE agency_subelement = '{AGENCY_SUBELEMENT}'
          AND occupational_series_code = '{SERIES_CODE}'
          AND snapshot_yyyymm BETWEEN '{START_YM}' AND '{END_YM}'
          AND appointment_type NOT IN {POLITICAL_EXCLUSION_SQL}
        GROUP BY 1, 2
    """).df()

## Query employment

Pulls every available month Nov 2024–present by default
(`EMPLOYMENT_MONTH_STRIDE = 1`). Employment snapshots are the full federal
workforce each month (26–75 MiB each) filtered down to ~6,000 AUSA rows --
raise the stride if you widen the date range back toward full history and
want to keep the query fast.

In [ ]:
employment_interval_desc = (
    "every available month" if EMPLOYMENT_MONTH_STRIDE == 1
    else f"every {EMPLOYMENT_MONTH_STRIDE} months"
)
print(f"Querying employment (headcount), {employment_interval_desc}...")
df_employment = query_employment()
print(f"  {len(df_employment)} (month, area) rows, {df_employment.ym.nunique()} distinct months")

## Table

[great_tables](https://posit-dev.github.io/great-tables/) instead of a
matplotlib heatmap — a color-shaded table reads more clearly than an
`imshow` grid at this size, with real numbers in every cell instead of tiny
rotated-axis labels. One row per month, DC and rest-of-country as separate
columns, each also shown indexed to the Nov 2024 baseline (=100%) so loss
reads as a percentage. Diverging red(loss)/blue(gain) color on the %
columns only — see `build_employment_gt`'s docstring for why the raw
counts are left uncolored.

In [ ]:
DIVERGING_PALETTE = ["#B2182B", "#F7F7F7", "#2166AC"]  # red (below baseline) - white - blue (above baseline)
ATTY_NOTE = "career attorneys only, series 0905 — excludes political appointees"
SOURCE_NOTE = (
    "Source: OPM/EHRI (impactproject/opm-ehri-data on HuggingFace), queried live. "
    "DC vs. rest-of-country only — finer geography (state/city) is privacy-redacted "
    "for this population."
)

# Fixed, generous domain for the % columns below -- NOT derived from this
# window's own min/max. A domain scaled to just this window's own extremes
# makes the worst month always look maximally saturated no matter how mild
# the real swing is (a 9% decline looked nearly as dark red as a 14% one
# when the domain was [85,100]). Anchoring to a fixed, meaningfully-extreme
# reference instead means color intensity reflects how bad things actually
# are, not just "worst cell inside whatever window I happen to be looking
# at right now."
HEADCOUNT_PCT_DOMAIN = [75, 125]  # full color at a +/-25pp swing from baseline headcount


def to_area_table(df, value_col):
    """One row per month, DC / rest-of-country as columns. No Total column
    -- rest-of-country outnumbers DC ~10:1, so Total just mirrors
    rest-of-country's pattern almost exactly and adds a third column of
    numbers without adding a third story. Month is formatted "Mon YYYY"
    (e.g. "Nov 2024") -- the raw "202411" YYYYMM string is hard to read.
    """
    pivot = df.pivot_table(index="ym", columns="area", values=value_col, aggfunc="sum")
    for c in ("DC", "Rest of country"):
        if c not in pivot.columns:
            pivot[c] = pd.NA
    tbl = pivot[["DC", "Rest of country"]].sort_index().reset_index().rename(columns={"ym": "Month"})
    tbl["Month"] = pd.to_datetime(tbl["Month"], format="%Y%m").dt.strftime("%b %Y")
    return tbl


def build_employment_gt(df):
    """Headcount table. Raw counts (DC/rest-of-country) are left UNCOLORED
    on purpose -- coloring them on their own scale while the % columns use
    a diverging scale told two contradictory stories with the same dark
    color (dark = high count = good, vs. dark = far from baseline = bad),
    right next to each other in the same row. Only the % columns carry
    color, both sharing ONE fixed domain (HEADCOUNT_PCT_DOMAIN), so the
    same percentage always renders as the same shade regardless of which
    area column it's in, DC's shade is directly comparable to
    rest-of-country's, and the color intensity means the same thing across
    different runs of this notebook.

    Count columns are labeled "Count", not "DC"/"Rest of country" again --
    the spanner above each pair already says that; repeating it under the
    spanner is redundant. cols_width keeps the table close to its content
    width instead of the browser/notebook stretching it to fill the cell.
    """
    tbl = to_area_table(df, "headcount")
    baseline_month = tbl.iloc[0]["Month"]
    count_cols = ["DC", "Rest of country"]
    idx_cols = [f"{c}_idx" for c in count_cols]
    for col in count_cols:
        tbl[f"{col}_idx"] = tbl[col] / tbl.iloc[0][col] * 100

    interval_desc = (
        "Every available month" if EMPLOYMENT_MONTH_STRIDE == 1
        else f"Every {EMPLOYMENT_MONTH_STRIDE} months"
    )
    gt = (
        GT(tbl)
        .tab_header(
            title=f"AUSA headcount ({ATTY_NOTE})",
            subtitle=f"{interval_desc}. % columns indexed to the {baseline_month} baseline (=100%), shared color scale",
        )
        .tab_source_note(SOURCE_NOTE)
        .fmt_integer(columns=count_cols)
        .fmt_number(columns=idx_cols, decimals=0, pattern="{x}%")
        .cols_label(**{c: "Count" for c in count_cols}, **{f"{c}_idx": "% of baseline" for c in count_cols})
        .cols_width({
            "Month": "90px",
            "DC": "70px", "DC_idx": "100px",
            "Rest of country": "110px", "Rest of country_idx": "100px",
        })
        .data_color(columns=idx_cols, palette=DIVERGING_PALETTE, domain=HEADCOUNT_PCT_DOMAIN)
    )
    for col in count_cols:
        gt = gt.tab_spanner(label=col, columns=[col, f"{col}_idx"])
    return gt

In [ ]:
build_employment_gt(df_employment)

## Caveats

- **Area is DC vs. rest-of-country only** — see the scope note at the top. This is a real limit of the public EHRI data (privacy suppression), not a limit of this notebook's queries.
- **Employment/headcount only, not accessions/separations** — see the scope note at the top for why (OPM's guidance: employment is authoritative when the two disagree, and they did — DC's Jan 2025 net was -30 by the accessions/separations math, but the actual headcount barely moved that month, with most of the drop landing in February's snapshot instead). The accessions/separations datasets exist in the same HF repo (`accessions/`, `separations/`, same schema pattern used here) if you want to reconstruct hires/departures yourself, but be aware of that mismatch before trusting a single-month number from them.
- **Employment defaults to every available month** (`EMPLOYMENT_MONTH_STRIDE = 1`). Within this ~20-month window that's fast (~20s) and doesn't trip HuggingFace's rate limit — that only happened pulling the FULL 2015-present history (~140 files) in an earlier version of this notebook. Raise the stride above 1 if you widen `START_YM`/`END_YM` back toward full history and want to keep the query fast; it's a fixed, deterministic interval (always the same months), not a statistical sample.
- **Everything is queried live** — figures may shift slightly as OPM/EHRI publishes revisions (files are versioned; this notebook always takes the latest version per month).
- **`count` is a string column in the source parquet** and is cast with `TRY_CAST` — any row that fails to cast contributes 0, not an error, so a malformed value would silently under-count rather than crash the notebook.